# DIMER posterior-predictive video

Experimental-vs-synthetic viewer for the posterior-predictive check. The engine
`Script_Bank/Analysis/SRM_AND_SBI_DIMER_ALP_DETECTOR_Posterior_Predictive_Video.py` renders a
synthetic video from an experimental recording's MAP imaging estimate and persists it
(`*_Synthetic_Video.npz`); this notebook only **views** the persisted clip. It never runs a
simulation, so scrubbing, zooming, and re-running cells never regenerate the trajectory. To get
a new trajectory, re-run the engine (a deliberate step), optionally with `--seed`.

**How to run.** This is a pure viewer -- it needs only `numpy`, `matplotlib`, and `ipywidgets`
(no project package, no `MACHINE_PROFILE`, no ReaDDy). Render a clip with the engine on the
machine that holds the data (e.g. rcl01), copy the resulting `*_Synthetic_Video.npz` to this
machine, set `CLIP_PATH` in the next code cell, and launch `jupyter lab`. The clip is
self-contained (experimental + synthetic + provenance), so it opens anywhere.

The synthetic's **motion** is a fresh RDS-nuisance draw (not the experimental track); the
comparison reads **imaging appearance** -- PSF, brightness, noise, flicker. The color scaling
matches the engine's static figure, so a given frame looks identical in the scrubber, the player,
and the static figure: by default (`full`) a shared full-range `[min, max]` window over both
panels is used, so identical intensities map to identical colors and nothing is clipped
(`autoscale` per-frame and `percentile` are also available). The data stay 16-bit; the estimator
was calibrated on the fixed 8-bit rescale.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, interact
# Pure viewer: needs only numpy + matplotlib + ipywidgets. No project package, no
# MACHINE_PROFILE, no ReaDDy -- rendering happens in the engine on the data machine.

In [ ]:
# Point CLIP_PATH at a *_Synthetic_Video.npz produced by the engine
# (SRM_AND_SBI_DIMER_ALP_DETECTOR_Posterior_Predictive_Video.py). Copy it to this machine
# first; it is self-contained (experimental + synthetic + provenance), so any absolute path works.
CLIP_PATH = "<absolute path to ..._Synthetic_Video.npz>"

# Display color scaling (data stay 16-bit; this only sets how the magma colormap is scaled).
# Matches the engine's static figure, so a given frame looks identical in the scrubber, the
# player, and the static figure:
#   "full"       -> a SHARED full-range [min, max] window over BOTH panels (default); identical
#                   intensities map to identical colors and nothing is clipped.
#   "autoscale"  -> each displayed frame stretched to its own min/max (per-frame).
#   "percentile" -> a whole-clip [min, p99.99] window (keeps essentially the whole range, cutting
#                   only the top-0.01% hot-pixel sliver).
NORM_MODE = "full"

d = np.load(CLIP_PATH, allow_pickle=True)
experimental, synth = d["experimental"], d["synth"]        # (n_frames, H, W) uint16
n_frames = int(d["n_frames"])
fps = round(1.0 / float(d["frame_time_seconds"]))
print(f"experimental {experimental.shape}  synth {synth.shape}  |  {fps} fps  |  "
      f"kind={str(d['kind'])} cell={int(d['cell'])}  MAP source={str(d['map_source'])}  seed={int(d['seed'])}")
print(f"ADU  experimental  min {int(experimental.min())} median {int(np.median(experimental))} max {int(experimental.max())}")
print(f"ADU  synth         min {int(synth.min())} median {int(np.median(synth))} max {int(synth.max())}")

# Shared full-range window over BOTH panels (used in 'full' mode), and the whole-clip
# [min, p99.99] per-panel window (used in 'percentile' mode).
_FULL = (float(min(experimental.min(), synth.min())), float(max(experimental.max(), synth.max())))
_PCTL = {"experimental": (float(experimental.min()), float(np.percentile(experimental, 99.99))),
         "synth": (float(synth.min()), float(np.percentile(synth, 99.99)))}

def clim(which, frame2d=None):
    """(vmin, vmax) for a panel ('experimental'/'synth'): the shared full-range [min, max] over
    both panels in 'full' mode; the whole-clip [min, p99.99] window in 'percentile' mode;
    or the displayed frame's own min/max (per-frame) in 'autoscale' mode."""
    if NORM_MODE == "full":
        return _FULL
    if NORM_MODE == "percentile":
        return _PCTL[which]
    return (float(frame2d.min()), float(frame2d.max())) if frame2d is not None else (None, None)

print(f"NORM_MODE={NORM_MODE}  " + (
    f"shared full-range window: {tuple(round(x) for x in _FULL)}" if NORM_MODE == "full"
    else (f"percentile window: experimental {tuple(round(x) for x in _PCTL['experimental'])}  "
          f"synth {tuple(round(x) for x in _PCTL['synth'])}" if NORM_MODE == "percentile"
          else "(each displayed frame stretched to its own min/max)")))

In [ ]:
# Scrub frames and zoom a shared region-of-interest into BOTH panels together.
# zoom = 1 shows the full frame; higher zoom crops tighter around (center x, center y).
H, W = experimental.shape[1], experimental.shape[2]

def _roi(frame, cx, cy, zoom):
    half_y, half_x = int(H / (2 * zoom)), int(W / (2 * zoom))
    y0, y1 = max(0, cy - half_y), min(H, cy + half_y)
    x0, x1 = max(0, cx - half_x), min(W, cx + half_x)
    return frame[y0:y1, x0:x1], (x0, x1, y0, y1)

def show(frame, cx, cy, zoom):
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    for a, arr, title, which in ((ax[0], experimental, "EXPERIMENTAL", "experimental"),
                                 (ax[1], synth, "SYNTH (MAP)", "synth")):
        crop, ext = _roi(arr[frame], cx, cy, zoom)
        vmin, vmax = clim(which, crop)
        a.imshow(crop, cmap="magma", origin="lower", interpolation="none",
                 vmin=vmin, vmax=vmax, extent=ext)
        a.set_title(f"{title}   frame {frame}/{n_frames - 1}   zoom x{zoom:g}")
    plt.tight_layout(); plt.show()

interact(
    show,
    frame=IntSlider(min=0, max=n_frames - 1, step=1, value=n_frames // 2, description="frame"),
    cx=IntSlider(min=0, max=W - 1, step=1, value=W // 2, description="center x"),
    cy=IntSlider(min=0, max=H - 1, step=1, value=H // 2, description="center y"),
    zoom=FloatSlider(min=1.0, max=8.0, step=0.5, value=1.0, description="zoom"),
);

## Playback (real time)

Play both clips side by side at the native frame rate, using the **same per-frame color scaling
as the scrubber** (so a given frame looks the same in both, and matches the static figure). Set
`PLAY_ZOOM` (and the center) to play a **cropped region** and check whether experimental and
synthetic coincide locally. The player embeds every frame, so a long clip builds a large widget
-- the embed limit is raised below; raise `PLAY_EVERY` to lighten/speed the build (it stays
real-time).

In [ ]:
import matplotlib as mpl
from matplotlib import animation
from IPython.display import HTML

mpl.rcParams["animation.embed_limit"] = 256   # MB; a 1000-frame 2-panel player exceeds the 20 MB default
PLAY_EVERY = 1                                 # stride: raise (2, 5, ...) to shrink the player; stays real-time

# Zoom for the player (same ROI convention as the scrubber): PLAY_ZOOM = 1 plays the full frame;
# raise it and set the center to play a cropped region and inspect local agreement.
PLAY_ZOOM = 1.0
PLAY_CX, PLAY_CY = W // 2, H // 2

def _pcrop(frame):
    return _roi(frame, PLAY_CX, PLAY_CY, PLAY_ZOOM)[0]

idx = list(range(0, n_frames, PLAY_EVERY))
fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
panels = []
for a, arr, title, which in ((ax[0], experimental, "EXPERIMENTAL", "experimental"),
                             (ax[1], synth, "SYNTH (MAP)", "synth")):
    c0 = _pcrop(arr[0]); vmin, vmax = clim(which, c0)
    im = a.imshow(c0, cmap="magma", origin="lower", interpolation="none", vmin=vmin, vmax=vmax)
    a.set_title(title); a.set_xticks([]); a.set_yticks([]); panels.append((im, arr, which))
plt.close(fig)   # suppress the static duplicate; the player below renders it

def _update(i):
    for im, arr, which in panels:
        cf = _pcrop(arr[i]); im.set_data(cf)
        im.set_clim(*clim(which, cf))   # per-frame in autoscale (matches the scrubber); fixed in percentile
    fig.suptitle(f"frame {i}/{n_frames - 1}   ({fps} fps)   zoom x{PLAY_ZOOM:g}")
    return [im for im, _, _ in panels]

anim = animation.FuncAnimation(fig, _update, frames=idx, interval=1000.0 * PLAY_EVERY / fps, blit=False)
HTML(anim.to_jshtml())